# 08 · Registro del modelo

> **BORRADOR para revisión del grupo.** Guía: `material-del-curso/kit/guia-validacion-registro-y-app.md`, sección 5.

En la etapa de optimización el **tracking** guardó *ejecuciones*: cada ensayo, con sus parámetros y métricas. Eso responde "¿qué probamos?". El **registro de modelos** responde otra pregunta: **"¿cuál es el modelo que usamos, y en qué versión?"**

| | Tracking | Registro |
|---|---|---|
| Guarda | Ejecuciones (muchas, de prueba) | Versiones del modelo (pocas, candidatas a producción) |
| Pregunta | ¿Qué probamos y qué dio? | ¿Qué modelo está en uso y qué versión? |
| Ejemplo | 40 ensayos de la búsqueda | `venta_cruzada_seguros` versión 1, alias `champion` |

Un **alias** (`champion`) es una etiqueta móvil: quien use el modelo pide "el champion" y no le importa qué número de versión es. Si mañana hay uno mejor, se mueve el alias y nadie toca el código.

## 0. Conexión con el seguimiento

In [1]:
import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    donde = configure_mlflow()
for aviso in avisos:
    print("Aviso:", aviso.message)
experimento = set_experiment("venta-cruzada-seguros-modelo-final")
print("MLflow:", donde, "| experimento:", experimento)

2026/11/05 19:11:44 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at c:\Users\Usuario\Downloads\Grupo2\.venv Proyecto Gruo2\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/11/05 19:11:46 INFO mlflow.tracking.fluent: Experiment with name 'venta-cruzada-seguros-modelo-final' does not exist. Creating a new experiment.


MLflow: local | experimento: venta-cruzada-seguros-modelo-final


> **Si ves `local`:** no hay conexión con Databricks (sin `.env`, sin red o cuota agotada). El registro funciona igual, pero en tu equipo (`mlflow_local.db`). Es el respaldo previsto.

## 1. Cargar el modelo final y los resultados de la prueba

Salen del notebook 07: el modelo en `models/final_pipeline.joblib` y las métricas en `reports/07_metricas_test.json`.

In [2]:
import joblib
import numpy as np
import pandas as pd

modelo_path, info_path = ROOT / "models" / "final_pipeline.joblib", ROOT / "reports" / "07_metricas_test.json"
modelo = joblib.load(modelo_path)
info = json.loads(info_path.read_text(encoding="utf-8"))
print("modelo y resultados cargados:", type(modelo).__name__, "| prueba abierta:", info["prueba"]["abierta_el"])

modelo y resultados cargados: Pipeline | prueba abierta: La prueba se abrió por primera vez el 2026-11-05 19:09.


## 2. Qué se registra

El modelo final **con lo que hace falta para confiar en él**:

- los **parámetros** y el **tipo de modelo**,
- las **métricas en la prueba** (la evaluación que se hizo una sola vez),
- una **firma** (qué columnas espera y de qué tipo) y un **ejemplo de entrada**,
- **etiquetas** que dicen de dónde salió.

> **Nota técnica.** MLflow guarda por defecto los modelos de scikit-learn en un formato que **rechaza** un `Pipeline` con preprocesamiento ("untrusted types"). Por eso el registro usa `serialization_format="cloudpickle"`. Los archivos *pickle* **pueden ejecutar código al cargarse**: carga solo modelos que guardaste tú o tu grupo, nunca uno de una fuente en la que no confíes.

In [3]:
import mlflow

NOMBRE_MODELO = "venta_cruzada_seguros"
# Columnas LIMPIAS que recibe el pipeline (salida de la etapa 3); el notebook 09 agrega la capa que traduce lo que escribe una persona.
COLUMNAS_ENTRADA = ["Age", "Gender", "Previously_Insured", "Vehicle_Age", "Vehicle_Damage", "Region_Code", "Annual_Premium", "Premium_is_minimum"]

train = pd.read_parquet(ROOT / "data" / "processed" / "train.parquet")
ejemplo = train[COLUMNAS_ENTRADA].head(5)
nombre_completo = registered_model_name(NOMBRE_MODELO)
prueba = info["prueba"]
metricas = prueba["metricas"]["modelo final (LightGBM)"]

with mlflow.start_run(run_name="modelo-final") as run:
    mlflow.log_params({"modelo": info["decisiones"]["modelo_final"], "parametros": info["decisiones"]["parametros"],
                       **{f"lgbm_{k}": v for k, v in info["decisiones"]["parametros_valores"].items()},
                       "variables": info["decisiones"]["variables"], "calibrar": info["decisiones"]["calibrar"],
                       "umbral": round(1 / 6, 4), "R": 5})
    mlflow.log_metrics({"test_costo_por_cliente": metricas["cost"], "test_pr_auc": metricas["pr_auc"], "test_roc_auc": metricas["roc_auc"],
                        "test_recall_compradores": metricas["recall_1"], "test_precision_compradores": metricas["precision_1"],
                        "test_porcentaje_contactado": metricas["contact_rate"], "test_log_loss": metricas["log_loss"],
                        "test_costo_baseline": prueba["metricas"]["baseline: llamar si hubo daño"]["cost"],
                        "test_ic95_costo_bajo": prueba["costo_por_cliente"]["modelo final (LightGBM)"]["ic95_bajo"],
                        "test_ic95_costo_alto": prueba["costo_por_cliente"]["modelo final (LightGBM)"]["ic95_alto"]})
    mlflow.set_tags({"etapa": "08_registro", "nota": "prueba abierta una sola vez en la etapa 7", "decididas_el": info["decisiones"]["decidido_el"],
                     "sin_canal_de_venta": "true", "con_gender": "true"})
    mlflow.sklearn.log_model(modelo, name="model", input_example=ejemplo, registered_model_name=nombre_completo,
                             serialization_format="cloudpickle")
    run_id = run.info.run_id
print("ejecución:", run_id, "| modelo registrado como:", nombre_completo)

2026/11/05 19:11:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ejecución: 8a5c2e2177644ed4afe613249f958957 | modelo registrado como: venta_cruzada_seguros


Successfully registered model 'venta_cruzada_seguros'.
Created version '1' of model 'venta_cruzada_seguros'.


## 3. Poner el alias `champion`

Busca la versión recién registrada y dale el alias. A partir de ahora cualquiera pide `models:/<nombre>@champion`.

In [4]:
cliente = mlflow.MlflowClient()
version = max(int(v.version) for v in cliente.search_model_versions(f"name = '{nombre_completo}'"))
cliente.set_registered_model_alias(nombre_completo, "champion", str(version))
print(f"{nombre_completo}: versión {version} -> alias 'champion'")

venta_cruzada_seguros: versión 1 -> alias 'champion'


## 4. Comprobación de ida y vuelta

Un registro que no se puede **recuperar** no sirve. Carga el modelo desde el registro y comprueba que predice **exactamente igual** que el modelo local.

In [5]:
recuperado = mlflow.sklearn.load_model(f"models:/{nombre_completo}@champion")
muestra = train[COLUMNAS_ENTRADA].head(2000)
iguales = np.allclose(modelo.predict_proba(muestra), recuperado.predict_proba(muestra))
assert iguales, "el modelo recuperado del registro no coincide con el original"
print("el modelo recuperado del registro predice igual que el original (2.000 clientes):", iguales)

el modelo recuperado del registro predice igual que el original (2.000 clientes): True


## 5. Guardar la constancia

Anota dónde quedó el modelo en `reports/08_registro.json` y completa `reports/08_registro.md`.

In [6]:
from datetime import datetime

constancia = {"tracking": donde, "experimento": experimento, "modelo_registrado": nombre_completo, "version": version, "alias": "champion",
              "run_id": run_id, "registrado_el": datetime.now().strftime("%Y-%m-%d %H:%M"), "columnas_entrada": COLUMNAS_ENTRADA}
(ROOT / "reports" / "08_registro.json").write_text(json.dumps(constancia, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(constancia, indent=2, ensure_ascii=False))

{
  "tracking": "local",
  "experimento": "venta-cruzada-seguros-modelo-final",
  "modelo_registrado": "venta_cruzada_seguros",
  "version": 1,
  "alias": "champion",
  "run_id": "8a5c2e2177644ed4afe613249f958957",
  "registrado_el": "2026-11-05 19:12",
  "columnas_entrada": [
    "Age",
    "Gender",
    "Previously_Insured",
    "Vehicle_Age",
    "Vehicle_Damage",
    "Region_Code",
    "Annual_Premium",
    "Premium_is_minimum"
  ]
}


## Qué sigue

El notebook 09 exporta el modelo `champion` a un archivo (`models/model.joblib`) que viaja con la app. **La app no consulta Databricks**: no se ponen credenciales en la nube, y el despliegue no depende de la cuota de la plataforma. El registro sigue siendo la **fuente de verdad**: el archivo siempre se exporta desde la versión `champion`.